### Bidirektionale Breitensuche beim 2x2x2 Rubiks-Cube

Der 2x2x2 Rubiks-Cube hat total 3'674'160 Zustände.
Jeder Zustand ist im max. 11 Drehungen zu erreichen.
Suchen wir die Lösung zu einem Zustand, genügt es also, von diesen Zustand und vom initialen Zustand
6 Drehungen tief zu suchen.
Mit maximal 6 Drehungen lassen sich nur 62360 erreichen.
Statt max. 3'674'160, brauchen wir nur ca. 2 x  62'360 Zustände zu besuchen.
Hier lohnt sich bidirektionale Breitensuche.


**Würfel-Zustände und Operationen**  
Die Hauptfarbe eines Würfelchens ist weiss oder gelb.
Ein  Würfelchen mit Hauptfarbe weiss oder gelb hat Orientierung 0.  
Rotiert man ein Würfelchen mit Orientierung 0 um 1x$120^{\circ}$ (2x$120^{\circ}$) im Uhrzeigersinn um die Achse
vom Würfelzentrum zur Ecke des Würfelchens, dann hat das Würfelchen Orientierung 1 (bez. 2).


Der Zustand des Würfels wird mit 2 Tupeln der Länge 8 beschreiben.
- Das 1. Tuple ist eine Permutation der Zahlen 0 bis 7.  
  `(3, 0, 1, 2, 4, 5, 6, 7)` ist so zu lesen:  
  Würfelchen 3 ist auf Position 0 (dort wo Würfelchen 0 im initialen Zustand ist), Würfelchen 0 ist auf Position 1, u.s.w.  
  Das oberste Stockwerk hat sich also im Uhrzeigersinn gedreht, 
  die Würfelchen des unteren Stockwerks bleiben auf ihren Positionen.
- Das 2. Tuple enthält die Zahlen 0,1 und 2.  
  `(1, 2, 2, 1, 0, 0, 0, 0)` gibt die Orientierung der Würfelchen an.
  
Zum Beispiel ist nach Drehen von R der Würfel im Zustand  
`(0, 2, 5, 3, 4, 6, 1, 7) + (0, 1, 2, 0, 0, 1, 2, 0)`.  

Das Modul `babycube` stellt eine Funktion `apply_op(op, state=ID)` bereit,
die eine Operation `op` auf den Zustand `state` anwendet und den resultiereden Zustand zurück gibt. 
Das heisst, auf den Zustand state wenden wir nun noch die Drehunen an, die
den Anfanszustand in den Zustand op überführen.

Ist z.B. op die Drehung R und state wurde erhalten durch Drehen von RU2F, dann
liefert  `apply_op(op, state)` den Zustand  RU2FR. 

Ist x eine der Drehungen in  {U,F,R, u,f, r, U2, R2, F2 } und
wissen wir, dass Zustand t die Form sx (x angewendet auf s) hat, können wir x zurückgewinnen, indem wir
t auf s^{-1} anwenden:

$t = sx$  
$s^{-1}t = s^{-1}s x = x$  

as Modul `babycube` stellt eine Funktion `inv_op(s)` bereit, welche den inversen Zustand $s^{-1}$ liefert.

Den 2x2x2 Würfel kann man auf jeder seiner 6 Seiten stellen, dann hat man noch 4 Möglichkeiten eine
Voderseite zu wählen. Wir wollen diese 24 Zustände nicht unterscheiden.
Deshalb positioniern wir den Würfel immer so, dass das Würfelchen mit den Seiten **orange, grün und gelb hinten links liegt, mit der gelben Seite nach unten. 


Lassen wir dieses  Würfelchen an seine Platz,
können wir noch die obere, vordere und rechte Seite drehen.
Eine Drehung im Uhrzeigersinn wird mit U (Up), F (Front) und R (Right) notiert,
eine Drehung im Gegenuhrzeigersinn mit u, f und r, und eine Doppeldrehung mit U2, F2 und R2,
siehe [Notation für Drehunen](https://www.jperm.net/3x3/moves).

Ein Wort der Form RU2F interpretieren wird als das Nacheinanderausführen der
Drehungen R, U2 und F.
Diese Operation können wir leicht rückgängig machen durch Drehen von fU2r. Insgesamt wird dann
RU2FfU2r ausgefüht, was letztlich den Würfel in der Ausgansposition belässt.

Jeder Zustand s des Würfels fassen wir auch als Operation aus: Die Drehungen, die wir ausführen müssen, um
diesen Zustand von Ausganszustand her zu erreichen.
Diese Drehungen können wir aber auch auf einen anderen Startzustand anwenden.
Zu jedem Zustand $s$ gibt es auch einen inversen Zustand $s^{-1}$:
Führt z.B. RU2F zum Zustand s, dann führt z.B. fU2r zum Zustand $s^{-1}$.

In [50]:
from babycube import ID, KEY_OP, OP_KEY, inv_key, inv_op, apply_op

ID

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

In [51]:
inv_key('R')

'r'

In [52]:
s = apply_op(KEY_OP['R'])
s

(0, 2, 5, 3, 4, 6, 1, 7, 0, 1, 2, 0, 0, 1, 2, 0)

In [53]:
apply_op(KEY_OP['r'], s)

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

***
Eine Folge von Drehungen (Operationen) geben wir als
Wort der Form `'RF2R2u'` an. Um diese Operationen rückgänging zu machen,
ist `'UR2F2r`' anzuwenden.
***

In [54]:
def get_turns(word):
    '''gibt die Drehungen  in word zurueck'''
    turns = []
    for c in word:
        if c != '2':
            turns.append(c)
        else:
            turns[-1] += '2'
    return turns


def apply_turn(k, s=ID):
    op = KEY_OP[k]
    s = apply_op(op, s)
    return s


def apply_turns(turns, s=ID):
    '''Fuehre eine Liste/Tuple von Operationen aus'''
    for k in turns:
        s = apply_turn(k, s)
    return s


def apply_word(word, s=ID):
    '''fuehre ein Wort aus'''
    for k in get_turns(word):
        s = apply_turn(k, s)
    return s


def inv_word(word):
    '''liefert Wort, welche umkekehrte Operation ausfuehrt'''
    keys = get_turns(word)
    return ''.join(inv_key(k) for k in reversed(keys))

In [55]:
word = 'RUFU2rf'
turns = get_turns(word)
turns

['R', 'U', 'F', 'U2', 'r', 'f']

In [56]:
state1 = apply_turns(turns)
state2 = apply_word(word)
state1 == state2, state1

(True, (5, 1, 3, 4, 0, 6, 2, 7, 0, 0, 2, 0, 1, 1, 2, 0))

In [57]:
word = 'RUF2U2rUrUR2fU2FurfU2'
s = apply_word(word)
s

(4, 1, 3, 5, 0, 2, 6, 7, 1, 0, 2, 0, 2, 0, 1, 0)

In [58]:
w = inv_word(word)
w

'U2FRUfU2FR2uRuRU2F2ur'

In [59]:
apply_word(w, s)

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

In [60]:
t = apply_turn('R', s)
t

(4, 3, 2, 5, 0, 6, 1, 7, 1, 0, 2, 0, 2, 2, 2, 0)

***
t = s x
s^{-1}t = s^{-1}s x = x
***

In [61]:
def get_transition(s, t):
    return OP_KEY.get(apply_op(t, inv_op(s)))

In [62]:
word = 'RUF2U2rUrUR2fU2FurfU2'
s = apply_word(word)
t = apply_turn('R', s)

get_transition(s, t)

'R'

***
Unsere Funktion `get_neighbors_1(state)` könnte zu jedem neuen Zustand auch noch angegen, durch welche Drehung
er aus dem vorangehenden Zustand hervorgeht.
Dann kann man die Lösung leicht aus dem Pfad herauslesen.  
Um die in Modul die Funktonen aus dem Modul `search_strategies` direkt anwenden zu können,
`get_neighbors(state)` nur den Zustand zurück. Die Drehung, welche den alten in den neuen Zustand überführt lässt sich aber leicht berechnen.
***

In [63]:
import search_strategies as S


def get_neighbors_1(state):
    '''gibt Name der Operation und den neues Zustand zurueck'''
    for turn in ('F', 'R', 'U', 'F2', 'f', 'R2', 'r', 'U2', 'u'):
        new_state = apply_turn(turn, state)
        yield turn, new_state


def get_neighbors(state):
    '''gibt Name der Operation und den neues Zustand zurueck'''
    for turn in ('F', 'R', 'U', 'F2', 'f', 'R2', 'r', 'U2', 'u'):
        new_state = apply_turn(turn, state)
        yield new_state


def word_to_goal(path_home):
    turns = [get_transition(s, t) for t, s in zip(path_home[:-1], path_home[1:])]
    return ''.join(turns[::-1])

In [64]:
list(get_neighbors_1(ID))

[('F', (0, 1, 3, 4, 5, 2, 6, 7, 0, 0, 1, 2, 1, 2, 0, 0)),
 ('R', (0, 2, 5, 3, 4, 6, 1, 7, 0, 1, 2, 0, 0, 1, 2, 0)),
 ('U', (3, 0, 1, 2, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)),
 ('F2', (0, 1, 4, 5, 2, 3, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)),
 ('f', (0, 1, 5, 2, 3, 4, 6, 7, 0, 0, 1, 2, 1, 2, 0, 0)),
 ('R2', (0, 5, 6, 3, 4, 1, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0)),
 ('r', (0, 6, 1, 3, 4, 2, 5, 7, 0, 1, 2, 0, 0, 1, 2, 0)),
 ('U2', (2, 3, 0, 1, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)),
 ('u', (1, 2, 3, 0, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0))]

In [65]:
list(get_neighbors(ID))

[(0, 1, 3, 4, 5, 2, 6, 7, 0, 0, 1, 2, 1, 2, 0, 0),
 (0, 2, 5, 3, 4, 6, 1, 7, 0, 1, 2, 0, 0, 1, 2, 0),
 (3, 0, 1, 2, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0),
 (0, 1, 4, 5, 2, 3, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0),
 (0, 1, 5, 2, 3, 4, 6, 7, 0, 0, 1, 2, 1, 2, 0, 0),
 (0, 5, 6, 3, 4, 1, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0),
 (0, 6, 1, 3, 4, 2, 5, 7, 0, 1, 2, 0, 0, 1, 2, 0),
 (2, 3, 0, 1, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0),
 (1, 2, 3, 0, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)]

***
Wir nutzen Breitensuche, um alle Zustände die in 6 Drehungen erreichbar sind zu finden.  
**Beachte**: Sobald der erste Knoten mit Tiefe 6 von rechts her aus dem Deck entfernt wurde,
befinden sich nur noch Knoten der Tiefe $6$ im Deque und zwar bereits alle Knoten dieser Tiefe.
***

In [78]:
start = ID
goal = None

state, go_back, dist_dict = S.search_bf(start, get_neighbors, goal, max_depth=6)
state, len(go_back)

Failure. Count: 62360


((0, 2, 6, 5, 1, 4, 3, 7, 2, 0, 1, 0, 1, 2, 0, 0), 62360)

In [79]:
depth_nstates = {}
for state, depth in dist_dict.items():
    depth_nstates[depth] = depth_nstates.get(depth, 0) + 1

depth_nstates

{0: 1, 1: 9, 2: 54, 3: 321, 4: 1847, 5: 9992, 6: 50136}

In [80]:
state

(0, 2, 6, 5, 1, 4, 3, 7, 2, 0, 1, 0, 1, 2, 0, 0)

In [81]:
path = S.get_path_home(state, go_back)
path

[(0, 2, 6, 5, 1, 4, 3, 7, 2, 0, 1, 0, 1, 2, 0, 0),
 (6, 5, 0, 2, 1, 4, 3, 7, 1, 0, 2, 0, 1, 2, 0, 0),
 (6, 5, 2, 1, 4, 0, 3, 7, 1, 0, 1, 0, 0, 1, 0, 0),
 (6, 2, 0, 1, 4, 3, 5, 7, 1, 2, 0, 0, 0, 1, 2, 0),
 (1, 6, 2, 0, 4, 3, 5, 7, 0, 1, 2, 0, 0, 1, 2, 0),
 (1, 2, 3, 0, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0),
 (0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)]

In [82]:
scramble = word_to_goal(path)
scramble

'ururfU2'

In [83]:
apply_word(scramble) == state

True

In [84]:
solution = inv_word(scramble)
solution

'U2FRURU'

In [85]:
apply_word(solution, state)

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)

### Aufgabe
Benutze folgende Funktion, um den Würfel zu verdrehen.
```python
import random


def scramble_cube(n=100):
    ops = random.choices(list(cube.OPS), k=n)
    return apply_ops(ops)
```
Verwende bidirektionale Breitensuche, um den Weg von einem verdrehten Würfel zum initialen Zustand zu finden.

In [86]:
s = apply_word('RUF2U2rUrUR2fU2FurfU2')
s

(4, 1, 3, 5, 0, 2, 6, 7, 1, 0, 2, 0, 2, 0, 1, 0)

In [87]:
midpoint, go_backs = S.search_bibf(ID, get_neighbors, s)
path = S.join_paths_home(midpoint, go_backs)
scramble = word_to_goal(path)
scramble

Success. Count: 1042


'F2RF2urF2RF2r'

In [88]:
solution = inv_word(scramble)
solution

'RF2rF2RUF2rF2'

In [89]:
apply_word(solution, s)

(0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0)